# Generate speech with SLM

Load a native safetensors checkpoint, continue a short unit prompt, and decode the generated 25 Hz mHuBERT K=500 units with the matching CodeHiFiGAN vocoder. Set `MODEL_ID` to a local staged directory or a published Hugging Face model ID.

In [ ]:
import os
import torch
import soundfile as sf
from IPython.display import Audio, display
from slm import TransformerLM
from slm.audio import load_codehifigan, decode_units

In [ ]:
MODEL_ID = os.environ.get("SLM_MODEL_ID", "tiagoCuervo/gslm-scaling-155m-65p3b")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerLM.from_pretrained(MODEL_ID, device=device).eval()

The prompt below is a short example unit sequence. For meaningful continuation, replace it with units extracted from speech using the checkpoint's `unit_extractor` metadata.

In [ ]:
prompt = torch.tensor([[500]], device=device)
torch.manual_seed(7)
sequence = model.generate(prompt, max_new_tokens=250, temperature=0.7, top_k=50)
generated_units = sequence[0, prompt.shape[1]:].cpu()

In [ ]:
vocoder = load_codehifigan(device=device, vocab_size=500)
waveform = decode_units(generated_units, vocoder, vocab_size=500)
sf.write("slm_generation.wav", waveform, 16_000)
display(Audio(waveform, rate=16_000))